# Flower Detection — YOLOv7-tiny benchmark

Upstream YOLOv7 is pinned to torch 2.5.1: its `train.py` calls `torch.load()` on pickled checkpoints,
which torch 2.6+ rejects by default (`weights_only=True`). Train 30 epochs, then the script keeps the
epoch-14 and epoch-29 weights (epochs 15 and 30) and evaluates both on the test split.

**Install cell restarts the runtime** — after it finishes, continue from the clone cell.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

WORK = Path("/content") if Path("/content").is_dir() else Path("/kaggle/working")
IN_COLAB = WORK == Path("/content")
PROJECT = WORK / "flower_detect"
REPO = "https://github.com/hit1363/flower_detect.git"
DATA_YAML = PROJECT / "dataset/Annotated/split/data.yaml"
# Annotated images exported from the team Drive folder (images/, labels/, metadata.csv).
DRIVE_SOURCE = Path("/content/drive/MyDrive/flower_detection/02_annotated")
DEVICE, BATCH, WORKERS = "0", 8, 2
print(f"platform={'colab' if IN_COLAB else 'kaggle'} project={PROJECT}")

In [ ]:
# Pinned stack for upstream YOLOv7; skip the torch line if Colab already ships torch 2.5.x.
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy==1.26.4 tensorboard thop pyyaml
import os
os.kill(os.getpid(), 9)  # restart so the new torch is loaded

In [ ]:
import torch
print(torch.__version__)
assert torch.cuda.is_available(), "Select a GPU runtime."
assert torch.__version__.startswith("2.5."), f"YOLOv7 needs torch 2.5.x, found {torch.__version__}"
REPO_7 = WORK / "yolov7"
if not REPO_7.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/WongKinYiu/yolov7", str(REPO_7)], check=True)
if not (REPO_7 / "yolov7-tiny.pt").is_file():
    subprocess.run(["wget", "-q", "-O", str(REPO_7 / "yolov7-tiny.pt"),
                    "https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-tiny.pt"], check=True)
if not PROJECT.exists():
    subprocess.run(["git", "clone", "-q", REPO, str(PROJECT)], check=True)
print(f"yolov7 at {REPO_7}, project at {PROJECT}")

In [ ]:
# Pull the annotated source from Drive once, then validate and split it locally.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
SOURCE = PROJECT / "dataset/Annotated/source"
if DRIVE_SOURCE.is_dir():
    for folder in ("images", "labels"):
        shutil.copytree(DRIVE_SOURCE / folder, SOURCE / folder, dirs_exist_ok=True)
    shutil.copy2(DRIVE_SOURCE / "metadata.csv", SOURCE / "metadata.csv")
    print(f"Copied annotated source from {DRIVE_SOURCE}")
else:
    print(f"No Drive source at {DRIVE_SOURCE}; using {SOURCE} as already synced.")

In [ ]:
# validate first, then split; split refuses to overwrite an existing split on purpose
subprocess.run([sys.executable, "scripts/prepare_dataset.py", "validate"], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "scripts/prepare_dataset.py", "split"], cwd=PROJECT, check=True)

In [ ]:
subprocess.run([sys.executable, "scripts/train_yolov7.py", "--repo", str(WORK / "yolov7"),
                "--data-yaml", str(DATA_YAML), "--weights", str(WORK / "yolov7/yolov7-tiny.pt"),
                "--device", DEVICE, "--preflight"], cwd=PROJECT, check=True)

In [ ]:
run_dir = PROJECT / "supplementary" / "runs" / "yolov7"
if (run_dir / "results.csv").is_file():
    print("yolov7: already evaluated, skipping")
else:
    subprocess.run([sys.executable, "scripts/train_yolov7.py", "--repo", str(WORK / "yolov7"),
                    "--data-yaml", str(DATA_YAML), "--weights", str(WORK / "yolov7/yolov7-tiny.pt"),
                    "--device", DEVICE, "--batch", str(BATCH), "--workers", str(WORKERS)],
                   cwd=PROJECT, check=True)

In [ ]:
import pandas as pd
results = pd.read_csv(PROJECT / "supplementary/benchmark_results.csv")
print(f"{len(results)} of 12 rows recorded")
results

In [ ]:
import zipfile
archive = WORK / "flower_supplementary.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted((PROJECT / "supplementary").rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(PROJECT))
print(f"Packed {archive} ({archive.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    target = Path("/content/drive/MyDrive/flower_detection")
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, target / archive.name)
    shutil.copy2(PROJECT / "supplementary/benchmark_results.csv", target / "benchmark_results.csv")
    print(f"Copied results to {target}")